# Maven Dinner - Profile Processing Notebook

This notebook processes LinkedIn profiles and dinner registration data to create the final dataset for the interactive graph and guessing game.

In [1]:
import json
import pandas as pd
from pathlib import Path
import copy

## Cell 1: Load & Clean LinkedIn Profiles

Parse JSON and drop ONLY these fields:
- Top-level: `languages`, `skills`, `certifications`
- From `basic_info`: `creator_hashtags`, `is_creator`, `is_influencer`, `is_premium`, `open_to_work`, `created_timestamp`, `show_follower_count`, `background_picture_url`, `urn`, `top_skills`, `email`

In [2]:
# Load LinkedIn profiles
with open('dataset_linkedin_profiles.json', 'r', encoding='utf-8') as f:
    linkedin_profiles = json.load(f)

print(f"Loaded {len(linkedin_profiles)} LinkedIn profiles")

# Fields to drop from top-level
TOP_LEVEL_DROP = ['languages', 'skills', 'certifications']

# Fields to drop from basic_info
BASIC_INFO_DROP = [
    'creator_hashtags', 'is_creator', 'is_influencer', 'is_premium',
    'open_to_work', 'created_timestamp', 'show_follower_count',
    'background_picture_url', 'urn', 'top_skills', 'email'
]

def clean_profile(profile):
    """Remove specified fields from a profile."""
    cleaned = copy.deepcopy(profile)
    
    # Drop top-level fields
    for field in TOP_LEVEL_DROP:
        cleaned.pop(field, None)
    
    # Drop basic_info fields
    if 'basic_info' in cleaned:
        for field in BASIC_INFO_DROP:
            cleaned['basic_info'].pop(field, None)
    
    return cleaned

# Clean all profiles
cleaned_profiles = [clean_profile(p) for p in linkedin_profiles]

# Show sample of what fields remain
print("\nSample profile keys:", list(cleaned_profiles[0].keys()))
print("Sample basic_info keys:", list(cleaned_profiles[0].get('basic_info', {}).keys()))

Loaded 22 LinkedIn profiles

Sample profile keys: ['profileUrl', 'basic_info', 'experience', 'education', 'projects', 'recommendations']
Sample basic_info keys: ['fullname', 'first_name', 'last_name', 'headline', 'public_identifier', 'profile_url', 'profile_picture_url', 'about', 'location', 'follower_count', 'connection_count', 'current_company', 'current_company_urn']


## Cell 2: Load & Merge CSV Data

Extract columns: `name`, `A one liner...`, `If this dinner is wildly successful...` from dinner_list_reg.csv

In [6]:
# Load dinner registration CSV
dinner_df = pd.read_csv('dinner_list_reg.csv')

print(f"Loaded {len(dinner_df)} dinner registrations")
print("\nColumns:", dinner_df.columns.tolist())

# Extract relevant columns
one_liner_col = "A one liner on what you're building or working on right now."
want_to_meet_col = "If this dinner is wildly successful, you walk away having met someone who ______ (e.g., domain expertise, specific capability, stage experience, geography access, distribution channel, technical depth, capital perspective, mindset, etc.)"

# Create lookup dictionary by name (normalized)
dinner_data = {}
for _, row in dinner_df.iterrows():
    name = str(row['name']).strip()
    dinner_data[name.lower()] = {
        'one_liner': str(row.get(one_liner_col, '')).strip() if pd.notna(row.get(one_liner_col)) else '',
        'who_they_want_to_meet': str(row.get(want_to_meet_col, '')).strip() if pd.notna(row.get(want_to_meet_col)) else ''
    }

print(f"\nDinner data lookup created for {len(dinner_data)} people")
print("\nSample entries:")
for name, data in list(dinner_data.items())[:3]:
    print(f"  {name}: {data}")

Loaded 17 dinner registrations

Columns: ['api_id', 'name', 'first_name', 'last_name', 'email', 'phone_number', 'created_at', 'approval_status', 'checked_in_at', 'custom_source', 'qr_code_url', 'amount', 'amount_tax', 'amount_discount', 'currency', 'coupon_code', 'eth_address', 'solana_address', 'survey_response_rating', 'survey_response_feedback', 'ticket_type_id', 'ticket_name', 'LinkedIn profile', "A one liner on what you're building or working on right now.", 'If this dinner is wildly successful, you walk away having met someone who ______ (e.g., domain expertise, specific capability, stage experience, geography access, distribution channel, technical depth, capital perspective, mindset, etc.)']

Dinner data lookup created for 17 people

Sample entries:
  hrishi olickel: {'one_liner': 'southbridge.ai', 'who_they_want_to_meet': 'is bone-headed about solving something specific'}
  mokshya wadhwa: {'one_liner': 'Invited by arman', 'who_they_want_to_meet': 'Founder in FMCG with good on

## Cell 3: Build Full Profile JSON

Merge LinkedIn data with dinner registration data

In [7]:
def get_name_from_profile(profile):
    """Extract the full name from a profile."""
    basic_info = profile.get('basic_info', {})
    return basic_info.get('fullname', '').strip()

def find_dinner_data(name):
    """Find dinner registration data for a person by name."""
    name_lower = name.lower().strip()
    
    # Direct match
    if name_lower in dinner_data:
        return dinner_data[name_lower]
    
    # Try matching by first name or partial name
    for dinner_name, data in dinner_data.items():
        # Check if one name contains the other or first names match
        if name_lower in dinner_name or dinner_name in name_lower:
            return data
        # Check first name match
        if name_lower.split()[0] == dinner_name.split()[0]:
            return data
    
    return {'one_liner': '', 'who_they_want_to_meet': ''}

# Build merged profiles
merged_profiles = []
for profile in cleaned_profiles:
    name = get_name_from_profile(profile)
    dinner_info = find_dinner_data(name)
    
    # Add dinner registration fields to profile
    profile_with_dinner = copy.deepcopy(profile)
    profile_with_dinner['one_liner'] = dinner_info['one_liner']
    profile_with_dinner['who_they_want_to_meet'] = dinner_info['who_they_want_to_meet']
    
    merged_profiles.append(profile_with_dinner)

print(f"Merged {len(merged_profiles)} profiles with dinner data")

# Show merge results
print("\nMerge results:")
for p in merged_profiles:
    name = get_name_from_profile(p)
    has_dinner = bool(p['one_liner'] or p['who_they_want_to_meet'])
    print(f"  {name}: {'✓' if has_dinner else '✗'} dinner data")

Merged 22 profiles with dinner data

Merge results:
  Raman Gupta: ✗ dinner data
  Hrishi Olickel: ✓ dinner data
  Mokshya Wadhwa: ✓ dinner data
  Siddharth Shetty: ✓ dinner data
  Armaan Dhanda: ✓ dinner data
  Scott Sofian: ✓ dinner data
  Shrivardhan Goenka: ✓ dinner data
  Allison Law: ✓ dinner data
  Yajat Gulati: ✓ dinner data
  Canaan Poh: ✓ dinner data
  Carlo Charles: ✓ dinner data
  Cody T.: ✓ dinner data
  Samyak Baid: ✓ dinner data
  Sparsh ⠀: ✓ dinner data
  Suveen Ellawela: ✓ dinner data
  Umang Gupta: ✓ dinner data
  Kai Ze Tam: ✓ dinner data
  Manya Gupta: ✗ dinner data
  Yuv Bindal: ✗ dinner data
  Conrad Lin: ✗ dinner data
  🦄 kai chen: ✗ dinner data
  Sumit Jain: ✗ dinner data


## Cell 4: Export Scoped Profiles

Save as `scoped_profiles.json` for agent consumption

In [8]:
# Save scoped profiles for agent scripts
with open('scoped_profiles.json', 'w', encoding='utf-8') as f:
    json.dump(merged_profiles, f, indent=2, ensure_ascii=False)

print(f"Saved {len(merged_profiles)} profiles to scoped_profiles.json")
print(f"File size: {Path('scoped_profiles.json').stat().st_size / 1024:.1f} KB")

Saved 22 profiles to scoped_profiles.json
File size: 533.4 KB


---

## After Running Agent Scripts

Run the following cells AFTER executing:
1. `python generate_link_data.py` 
2. `python generate_game_data.py`

## Cell 5: Import Link Data

Load `link_data.json` after running agent script

In [9]:
# Load link data from agent output
try:
    with open('link_data.json', 'r', encoding='utf-8') as f:
        link_data = json.load(f)
    print(f"Loaded {len(link_data)} link entries")
    
    # Show sample
    if link_data:
        print("\nSample link entry:")
        print(json.dumps(link_data[0], indent=2))
except FileNotFoundError:
    print("link_data.json not found. Run generate_link_data.py first.")
    link_data = []

Loaded 231 link entries

Sample link entry:
{
  "person_a_name": "Raman Gupta",
  "person_b_name": "Hrishi Olickel",
  "synergies": [
    "AI-first builders based in Singapore",
    "Strong technical depth: ML, data, product",
    "Founder mindset; building domain-specific AI tools"
  ],
  "collaboration_opportunities": [
    "Advisor mentorship on product/engineering scaling",
    "Data/LLM infrastructure partnership for CaseCraft",
    "Intro swaps: SG founder community, early pilots"
  ],
  "why_a_should_meet_b": "Hrishi has scaled AI products as a YC startup CTO and can help you avoid common traps in building, shipping, and iterating CaseCraft. He\u2019s also Singapore-based and could plug you into a serious builder network.",
  "why_b_should_meet_a": "Raman is deeply technical and unusually focused on a concrete wedge (clinical training simulation) that could become a strong vertical AI product. He may become a design partner/customer for SouthbridgeAI\u2019s data layer and brings

## Cell 6: Import Game Data

Load `game_data.json` after running agent script

In [10]:
# Load game data from agent output
try:
    with open('game_data.json', 'r', encoding='utf-8') as f:
        game_data = json.load(f)
    print(f"Loaded game data for {len(game_data)} people")
    
    # Show sample
    if game_data:
        print("\nSample game entry:")
        print(json.dumps(game_data[0], indent=2))
except FileNotFoundError:
    print("game_data.json not found. Run generate_game_data.py first.")
    game_data = []

Loaded game data for 22 people

Sample game entry:
{
  "person_name": "Raman Gupta",
  "obscure_facts": [
    "This person once piloted an Excel-based \u201ccompany rivalry\u201d simulator and led a 30-person team for a high-school competition with 120+ participants.",
    "This person built a one-level 2D platformer called \u201cSwitcheroo\u201d featuring a shooting mechanic and time-altering puzzles\u2014and created the story, visuals, UI, animations, and level design themselves.",
    "This person previously interned in Beijing and split time between building a time-tracking web app, doing video editing, and helping with marketing.",
    "This person was part of a university Comedy Club (alongside their other campus commitments).",
    "This person helped develop a sports-betting ML \u201cstrategizer\u201d focused on finding arbitrage and even researching \u201ctime-arbitrage\u201d opportunities in single-bookie markets."
  ],
  "fun_facts": [
    "This person\u2019s favorite games 

## Cell 7: Build Final Dataset

Merge all data into `final_data.json`

In [11]:
def normalize_name(name):
    """Normalize name for matching."""
    return name.lower().strip()

# Create lookup for game data by name
game_data_lookup = {}
for entry in game_data:
    name = normalize_name(entry.get('person_name', ''))
    game_data_lookup[name] = {
        'hints': entry.get('obscure_facts', []),
        'fun_facts': entry.get('fun_facts', [])
    }

# Create lookup for link data - indexed by both person names
link_data_lookup = {}
for entry in link_data:
    name_a = normalize_name(entry.get('person_a_name', ''))
    name_b = normalize_name(entry.get('person_b_name', ''))
    
    # Store for person A -> person B
    if name_a not in link_data_lookup:
        link_data_lookup[name_a] = []
    link_data_lookup[name_a].append({
        'target_name': entry.get('person_b_name', ''),
        'score': entry.get('link_strength_score', 0),
        'synergies': entry.get('synergies', []),
        'collaboration_opportunities': entry.get('collaboration_opportunities', []),
        'why_x_should_meet_y': entry.get('why_a_should_meet_b', ''),
        'why_y_should_meet_x': entry.get('why_b_should_meet_a', ''),
        'one_liner_both': entry.get('one_liner_both', '')
    })
    
    # Store for person B -> person A (reverse)
    if name_b not in link_data_lookup:
        link_data_lookup[name_b] = []
    link_data_lookup[name_b].append({
        'target_name': entry.get('person_a_name', ''),
        'score': entry.get('link_strength_score', 0),
        'synergies': entry.get('synergies', []),
        'collaboration_opportunities': entry.get('collaboration_opportunities', []),
        'why_x_should_meet_y': entry.get('why_b_should_meet_a', ''),  # Reversed
        'why_y_should_meet_x': entry.get('why_a_should_meet_b', ''),  # Reversed
        'one_liner_both': entry.get('one_liner_both', '')
    })

print(f"Link data lookup created for {len(link_data_lookup)} people")
print(f"Game data lookup created for {len(game_data_lookup)} people")

Link data lookup created for 22 people
Game data lookup created for 22 people


In [12]:
# Build final profiles list
final_profiles = []

for profile in merged_profiles:
    basic_info = profile.get('basic_info', {})
    name = basic_info.get('fullname', '').strip()
    name_normalized = normalize_name(name)
    
    # Build the final profile structure
    final_profile = {
        'name': name,
        'profile_picture_url': basic_info.get('profile_picture_url', ''),
        'headline': basic_info.get('headline', ''),
        'about': basic_info.get('about', ''),
        'location': basic_info.get('location', {}),
        'current_company': basic_info.get('current_company', ''),
        'follower_count': basic_info.get('follower_count', 0),
        'connection_count': basic_info.get('connection_count', 0),
        'profile_url': basic_info.get('profile_url', ''),
        'experience': profile.get('experience', []),
        'education': profile.get('education', []),
        'projects': profile.get('projects', []),
        'recommendations': profile.get('recommendations', {}),
        'honors': profile.get('honors', []),
        'featured': profile.get('featured', []),
        'volunteer': profile.get('volunteer', []),
        'publications': profile.get('publications', []),
        'patents': profile.get('patents', []),
        'courses': profile.get('courses', []),
        'organizations': profile.get('organizations', []),
        'one_liner': profile.get('one_liner', ''),
        'who_they_want_to_meet': profile.get('who_they_want_to_meet', ''),
        'game_data': game_data_lookup.get(name_normalized, {'hints': [], 'fun_facts': []}),
        'links': sorted(
            link_data_lookup.get(name_normalized, []),
            key=lambda x: x['score'],
            reverse=True
        )
    }
    
    final_profiles.append(final_profile)

# Create final data structure
final_data = {
    'profiles': final_profiles
}

print(f"Built final data with {len(final_profiles)} profiles")

Built final data with 22 profiles


In [13]:
# Save final data
with open('final_data.json', 'w', encoding='utf-8') as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print(f"Saved final_data.json")
print(f"File size: {Path('final_data.json').stat().st_size / 1024:.1f} KB")

# Summary
print("\n=== Summary ===")
print(f"Total profiles: {len(final_profiles)}")
profiles_with_game_data = sum(1 for p in final_profiles if p['game_data']['hints'])
profiles_with_links = sum(1 for p in final_profiles if p['links'])
print(f"Profiles with game data: {profiles_with_game_data}")
print(f"Profiles with links: {profiles_with_links}")

# Show sample profile
print("\nSample final profile (first person):")
sample = final_profiles[0]
print(f"  Name: {sample['name']}")
print(f"  Headline: {sample['headline'][:60]}...")
print(f"  One-liner: {sample['one_liner']}")
print(f"  Links count: {len(sample['links'])}")
print(f"  Game hints count: {len(sample['game_data']['hints'])}")

Saved final_data.json
File size: 1160.8 KB

=== Summary ===
Total profiles: 22
Profiles with game data: 22
Profiles with links: 22

Sample final profile (first person):
  Name: Raman Gupta
  Headline: Training better and more doctors with CaseCraft, the AI Trai...
  One-liner: 
  Links count: 21
  Game hints count: 5


## Verification Cell

Verify the final data structure is correct

In [14]:
# Verify data integrity
print("=== Data Verification ===")

# Check all profiles have required fields
required_fields = ['name', 'profile_picture_url', 'headline', 'one_liner', 'game_data', 'links']
for profile in final_profiles:
    missing = [f for f in required_fields if f not in profile]
    if missing:
        print(f"WARNING: {profile.get('name', 'Unknown')} missing fields: {missing}")

print(f"\nAll {len(final_profiles)} profiles have required fields ✓")

# Check link data symmetry
if link_data:
    total_links = sum(len(p['links']) for p in final_profiles)
    expected_links = len(link_data) * 2  # Each link appears twice (A->B and B->A)
    print(f"Total link entries: {total_links} (expected ~{expected_links})")

# List all names for reference
print("\nAll attendees:")
for i, p in enumerate(final_profiles, 1):
    print(f"  {i}. {p['name']}")

=== Data Verification ===

All 22 profiles have required fields ✓
Total link entries: 462 (expected ~462)

All attendees:
  1. Raman Gupta
  2. Hrishi Olickel
  3. Mokshya Wadhwa
  4. Siddharth Shetty
  5. Armaan Dhanda
  6. Scott Sofian
  7. Shrivardhan Goenka
  8. Allison Law
  9. Yajat Gulati
  10. Canaan Poh
  11. Carlo Charles
  12. Cody T.
  13. Samyak Baid
  14. Sparsh ⠀
  15. Suveen Ellawela
  16. Umang Gupta
  17. Kai Ze Tam
  18. Manya Gupta
  19. Yuv Bindal
  20. Conrad Lin
  21. 🦄 kai chen
  22. Sumit Jain
